# Antimicrobial Resistance (AMR) Visualization by Farm Type

This notebook visualizes antimicrobial resistance data from Ukrainian farm wastewater samples.

**Data Source**: AMR detection from metagenomics workflow  
**Farm Types**: Poultry (6 samples) vs Pig (18 samples)  
**Sample Types**: Native vs Enriched  
**Regions**: Zaporizhzhia, Zhytomyr, Kyiv, Chernivtsi, Vinnytsia, Mykolaiv Oblasts

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plotting style configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palette (consistent with taxonomy visualizations)
COLORS = {
    'Poultry': '#2ecc71',  # Green
    'Pig': '#e74c3c',      # Red
    'Native': '#3498db',   # Blue
    'Enriched': '#9b59b6'  # Purple
}

print("Setup complete!")

## 2. Load Data

In [ ]:
# For Google Colab, uncomment:
# from google.colab import files
# uploaded = files.upload()  # Upload the CSV files

# Load AMR data
amr_summary = pd.read_csv('amr_by_sample.csv')
amr_full = pd.read_csv('amr_full_data.csv')
gene_matrix = pd.read_csv('amr_gene_matrix.csv', index_col=0)
class_matrix = pd.read_csv('amr_resistance_class_matrix.csv', index_col=0)
mapping = pd.read_csv('barcode_farm_mapping.csv')

print(f"AMR summary shape: {amr_summary.shape}")
print(f"AMR full data shape: {amr_full.shape}")
print(f"Gene matrix shape: {gene_matrix.shape}")
print(f"Resistance class matrix shape: {class_matrix.shape}")
print(f"\nColumns in summary: {amr_summary.columns.tolist()}")

In [ ]:
# Data overview
print("=" * 60)
print("AMR DATA OVERVIEW")
print("=" * 60)
print(f"\nTotal gene detections: {amr_full.shape[0]:,}")
print(f"Unique resistance genes: {amr_full['Gene'].nunique()}")
print(f"Unique resistance classes: {amr_full['resistance_class'].nunique()}")
print(f"\nSamples with AMR data: {amr_summary['barcode'].nunique()} / 24")

print("\nSamples per farm type:")
print(amr_summary.groupby('farm_type')['barcode'].nunique())

amr_summary.head()

---
## 3. AMR Gene Counts by Farm Type

In [ ]:
# Total AMR detections by farm type
farm_totals = amr_summary.groupby('farm_type').agg({
    'total_gene_detections': 'sum',
    'unique_genes': 'sum',
    'barcode': 'count'
}).reset_index()
farm_totals.columns = ['farm_type', 'total_detections', 'total_unique_genes', 'n_samples']

# Normalize per sample
farm_totals['detections_per_sample'] = farm_totals['total_detections'] / farm_totals['n_samples']
farm_totals['genes_per_sample'] = farm_totals['total_unique_genes'] / farm_totals['n_samples']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Total detections
colors = [COLORS[ft] for ft in farm_totals['farm_type']]
axes[0].barh(farm_totals['farm_type'], farm_totals['total_detections'], color=colors)
axes[0].set_xlabel('Total AMR Gene Detections')
axes[0].set_title('Total AMR Detections by Farm Type', fontsize=14, fontweight='bold')
for i, v in enumerate(farm_totals['total_detections']):
    axes[0].text(v + 500, i, f'{v:,.0f}', va='center', fontsize=11)

# Per sample (normalized)
axes[1].barh(farm_totals['farm_type'], farm_totals['detections_per_sample'], color=colors)
axes[1].set_xlabel('AMR Detections per Sample')
axes[1].set_title('Average AMR Detections per Sample', fontsize=14, fontweight='bold')
for i, v in enumerate(farm_totals['detections_per_sample']):
    axes[1].text(v + 100, i, f'{v:,.0f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\nSummary:")
print(farm_totals.to_string(index=False))

---
## 4. Top Resistance Genes by Farm Type

In [ ]:
# Top genes by farm type
gene_by_farm = amr_full.groupby(['farm_type', 'Gene']).size().reset_index(name='count')

# Get top 15 for each farm type
top_poultry = gene_by_farm[gene_by_farm['farm_type'] == 'Poultry'].nlargest(15, 'count')
top_pig = gene_by_farm[gene_by_farm['farm_type'] == 'Pig'].nlargest(15, 'count')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Poultry
axes[0].barh(top_poultry['Gene'], top_poultry['count'], color=COLORS['Poultry'])
axes[0].set_xlabel('Detection Count')
axes[0].set_title('Top 15 Resistance Genes - Poultry Farms', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(top_poultry['count']):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# Pig
axes[1].barh(top_pig['Gene'], top_pig['count'], color=COLORS['Pig'])
axes[1].set_xlabel('Detection Count')
axes[1].set_title('Top 15 Resistance Genes - Pig Farms', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(top_pig['count']):
    axes[1].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Antibiotic Class Distribution by Sample

In [ ]:
# Resistance class counts per barcode
class_by_barcode = amr_full.groupby(['barcode', 'resistance_class']).size().reset_index(name='count')

# Get top resistance classes
top_classes = amr_full['resistance_class'].value_counts().head(8).index.tolist()

# Group others
class_by_barcode['class_display'] = class_by_barcode['resistance_class'].apply(
    lambda x: x if x in top_classes else 'Other'
)

# Aggregate
class_agg = class_by_barcode.groupby(['barcode', 'class_display'])['count'].sum().reset_index()

# Pivot for stacked bar
pivot = class_agg.pivot(index='barcode', columns='class_display', values='count').fillna(0)

# Sort by farm type
barcode_order = mapping.sort_values('farm_type')['barcode'].tolist()
valid_barcodes = [b for b in barcode_order if b in pivot.index]
pivot = pivot.reindex(valid_barcodes)

# Plot
fig, ax = plt.subplots(figsize=(16, 8))
pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.8)

ax.set_xlabel('Barcode (Sample)')
ax.set_ylabel('AMR Detection Count')
ax.set_title('Antibiotic Resistance Class Distribution per Sample', fontsize=16, fontweight='bold')
ax.legend(title='Antibiotic Class', bbox_to_anchor=(1.02, 1), loc='upper left')

# Color x-axis labels by farm type
farm_type_map = mapping.set_index('barcode')['farm_type'].to_dict()
for label in ax.get_xticklabels():
    barcode = label.get_text()
    color = COLORS.get(farm_type_map.get(barcode, ''), 'black')
    label.set_color(color)
    label.set_fontweight('bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nX-axis label colors: Green = Poultry, Red = Pig")

---
## 6. AMR Heatmap by Sample

In [ ]:
# Get top 20 genes by total count
top_genes = amr_full['Gene'].value_counts().head(20).index.tolist()
gene_matrix_top = gene_matrix.loc[top_genes].copy()

# Sort columns by farm type
barcode_order = mapping.sort_values('farm_type')['barcode'].tolist()
valid_barcodes = [b for b in barcode_order if b in gene_matrix_top.columns]
gene_matrix_top = gene_matrix_top[valid_barcodes]

# Log transform
gene_matrix_log = np.log10(gene_matrix_top + 1)

# Sort rows by total abundance
row_order = gene_matrix_log.sum(axis=1).sort_values(ascending=False).index
gene_matrix_log = gene_matrix_log.loc[row_order]

# Plot
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    gene_matrix_log,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Log10(Count + 1)'},
    linewidths=0.5
)

ax.set_title('Top 20 AMR Genes Across Samples', fontsize=16, fontweight='bold')
ax.set_xlabel('Barcode (Sample)')
ax.set_ylabel('Resistance Gene')

# Color x-axis labels by farm type
for label in ax.get_xticklabels():
    barcode = label.get_text()
    color = COLORS.get(farm_type_map.get(barcode, ''), 'black')
    label.set_color(color)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("X-axis colors: Green = Poultry, Red = Pig")

---
## 7. Differential AMR: Pig vs Poultry

In [ ]:
# Calculate mean gene frequency per farm type
gene_by_farm = amr_full.groupby(['farm_type', 'Gene']).size().reset_index(name='count')

# Get sample counts per farm type for normalization
samples_per_farm = amr_summary.groupby('farm_type')['barcode'].nunique()

# Normalize by sample count
gene_by_farm['normalized'] = gene_by_farm.apply(
    lambda x: x['count'] / samples_per_farm[x['farm_type']], axis=1
)

# Pivot
pivot = gene_by_farm.pivot(index='Gene', columns='farm_type', values='normalized').fillna(0)

# Calculate log2 fold change
pivot['log2FC'] = np.log2((pivot['Pig'] + 0.1) / (pivot['Poultry'] + 0.1))
pivot['mean_count'] = (pivot['Pig'] + pivot['Poultry']) / 2

# Filter to genes with sufficient counts
pivot_filtered = pivot[pivot['mean_count'] > 5].copy()

# Sort by fold change
pivot_sorted = pivot_filtered.sort_values('log2FC')

# Take top/bottom genes
n_genes = 15
top_bottom = pd.concat([pivot_sorted.head(n_genes), pivot_sorted.tail(n_genes)])

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

colors = [COLORS['Pig'] if x > 0 else COLORS['Poultry'] for x in top_bottom['log2FC']]
ax.barh(top_bottom.index, top_bottom['log2FC'], color=colors)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('Log2 Fold Change (Pig / Poultry)')
ax.set_title('Differential AMR Gene Abundance: Pig vs Poultry', fontsize=14, fontweight='bold')

# Add annotation
ax.text(0.02, 0.98, '\u2190 Enriched in Poultry', transform=ax.transAxes,
        color=COLORS['Poultry'], fontweight='bold', va='top')
ax.text(0.98, 0.98, 'Enriched in Pig \u2192', transform=ax.transAxes,
        color=COLORS['Pig'], fontweight='bold', va='top', ha='right')

plt.tight_layout()
plt.show()

---
## 8. Multi-Drug Resistance Analysis

In [ ]:
# Use unique_resistance_classes from summary
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By farm type
sns.boxplot(data=amr_summary, x='farm_type', y='unique_resistance_classes', 
            palette=COLORS, ax=axes[0])
sns.stripplot(data=amr_summary, x='farm_type', y='unique_resistance_classes', 
              color='black', alpha=0.6, size=8, ax=axes[0])
axes[0].set_title('Unique Antibiotic Classes per Sample\n(Multi-Drug Resistance Indicator)', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Farm Type')
axes[0].set_ylabel('Number of Unique Antibiotic Classes')

# By farm type and sample type
sns.boxplot(data=amr_summary, x='farm_type', y='unique_resistance_classes', 
            hue='sample_type', 
            palette={'Native': COLORS['Native'], 'Enriched': COLORS['Enriched']}, 
            ax=axes[1])
axes[1].set_title('Multi-Drug Resistance: Farm Type x Sample Type', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Farm Type')
axes[1].set_ylabel('Number of Unique Antibiotic Classes')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nMulti-Drug Resistance Summary:")
print(amr_summary.groupby(['farm_type', 'sample_type'])['unique_resistance_classes'].agg(['mean', 'std', 'min', 'max']).round(1))

In [ ]:
# Unique genes comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By farm type
sns.boxplot(data=amr_summary, x='farm_type', y='unique_genes', 
            palette=COLORS, ax=axes[0])
sns.stripplot(data=amr_summary, x='farm_type', y='unique_genes', 
              color='black', alpha=0.6, size=8, ax=axes[0])
axes[0].set_title('Unique Resistance Genes per Sample', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Farm Type')
axes[0].set_ylabel('Number of Unique Genes')

# By farm type and sample type
sns.boxplot(data=amr_summary, x='farm_type', y='unique_genes', 
            hue='sample_type', 
            palette={'Native': COLORS['Native'], 'Enriched': COLORS['Enriched']}, 
            ax=axes[1])
axes[1].set_title('Unique Genes: Farm Type x Sample Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Farm Type')
axes[1].set_ylabel('Number of Unique Genes')

plt.tight_layout()
plt.show()

---
## 9. AMR by Region (Oblast)

In [ ]:
# AMR counts by oblast and farm type
oblast_summary = amr_summary.groupby(['oblast', 'farm_type']).agg({
    'total_gene_detections': 'sum',
    'unique_genes': 'mean',
    'barcode': 'count'
}).reset_index()
oblast_summary.columns = ['oblast', 'farm_type', 'total_detections', 'avg_unique_genes', 'n_samples']

# Order oblasts by total detections
oblast_order = amr_summary.groupby('oblast')['total_gene_detections'].sum().sort_values(ascending=False).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total detections by oblast
sns.barplot(
    data=oblast_summary, 
    x='oblast', 
    y='total_detections', 
    hue='farm_type',
    palette=COLORS, 
    order=oblast_order,
    ax=axes[0]
)
axes[0].set_title('Total AMR Detections by Region', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Oblast (Region)')
axes[0].set_ylabel('Total AMR Detections')
axes[0].tick_params(axis='x', rotation=45)

# Average unique genes by oblast
sns.barplot(
    data=oblast_summary, 
    x='oblast', 
    y='avg_unique_genes', 
    hue='farm_type',
    palette=COLORS, 
    order=oblast_order,
    ax=axes[1]
)
axes[1].set_title('Average Unique AMR Genes by Region', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Oblast (Region)')
axes[1].set_ylabel('Avg Unique Genes per Sample')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Summary
print("\nAMR by Region:")
print(oblast_summary.to_string(index=False))

---
## 10. Resistance Class Heatmap

In [ ]:
# Sort columns by farm type
valid_barcodes = [b for b in barcode_order if b in class_matrix.columns]
class_matrix_sorted = class_matrix[valid_barcodes].copy()

# Filter to top classes
top_classes = class_matrix_sorted.sum(axis=1).nlargest(15).index
class_matrix_top = class_matrix_sorted.loc[top_classes]

# Log transform
class_matrix_log = np.log10(class_matrix_top + 1)

# Sort rows
row_order = class_matrix_log.sum(axis=1).sort_values(ascending=False).index
class_matrix_log = class_matrix_log.loc[row_order]

# Plot
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    class_matrix_log,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Log10(Count + 1)'},
    linewidths=0.5,
    annot=True,
    fmt='.1f'
)

ax.set_title('Antibiotic Resistance Class Distribution Across Samples', fontsize=16, fontweight='bold')
ax.set_xlabel('Barcode (Sample)')
ax.set_ylabel('Antibiotic Class')

# Color x-axis labels by farm type
for label in ax.get_xticklabels():
    barcode = label.get_text()
    color = COLORS.get(farm_type_map.get(barcode, ''), 'black')
    label.set_color(color)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("X-axis colors: Green = Poultry, Red = Pig")

---
## 11. Statistical Analysis

In [ ]:
# Statistical comparison: Mann-Whitney U test (non-parametric)
print("=" * 60)
print("STATISTICAL COMPARISON: PIG vs POULTRY")
print("Mann-Whitney U Test (non-parametric)")
print("=" * 60)

poultry = amr_summary[amr_summary['farm_type'] == 'Poultry']
pig = amr_summary[amr_summary['farm_type'] == 'Pig']

metrics = ['total_gene_detections', 'unique_genes', 'unique_resistance_classes']
metric_names = ['Total Gene Detections', 'Unique Genes', 'Unique Resistance Classes']

for metric, name in zip(metrics, metric_names):
    poultry_vals = poultry[metric].dropna()
    pig_vals = pig[metric].dropna()
    
    if len(poultry_vals) > 0 and len(pig_vals) > 0:
        stat, p_value = stats.mannwhitneyu(poultry_vals, pig_vals, alternative='two-sided')
        
        significance = ""
        if p_value < 0.001:
            significance = "***"
        elif p_value < 0.01:
            significance = "**"
        elif p_value < 0.05:
            significance = "*"
        else:
            significance = "ns"
        
        print(f"\n{name}:")
        print(f"  Poultry (n={len(poultry_vals)}): mean={poultry_vals.mean():.1f}, median={poultry_vals.median():.1f}")
        print(f"  Pig (n={len(pig_vals)}): mean={pig_vals.mean():.1f}, median={pig_vals.median():.1f}")
        print(f"  U-statistic: {stat:.2f}, p-value: {p_value:.4f} {significance}")

print("\n" + "-" * 60)
print("Significance levels: * p<0.05, ** p<0.01, *** p<0.001, ns = not significant")

---
## 12. Summary

In [ ]:
print("=" * 70)
print("AMR ANALYSIS SUMMARY")
print("=" * 70)

print(f"\nDATASET OVERVIEW")
print(f"   Total AMR detections: {amr_full.shape[0]:,}")
print(f"   Unique resistance genes: {amr_full['Gene'].nunique()}")
print(f"   Unique antibiotic classes: {amr_full['resistance_class'].nunique()}")
print(f"   Samples with AMR data: {amr_summary['barcode'].nunique()} / 24")

print(f"\nFARM TYPE COMPARISON")
for farm_type in ['Poultry', 'Pig']:
    subset = amr_summary[amr_summary['farm_type'] == farm_type]
    print(f"   {farm_type}:")
    print(f"     - Samples: {len(subset)}")
    print(f"     - Total detections: {subset['total_gene_detections'].sum():,}")
    print(f"     - Avg unique genes: {subset['unique_genes'].mean():.1f}")
    print(f"     - Avg resistance classes: {subset['unique_resistance_classes'].mean():.1f}")

print(f"\nTOP 10 RESISTANCE GENES")
top_genes = amr_full['Gene'].value_counts().head(10)
for i, (gene, count) in enumerate(top_genes.items(), 1):
    print(f"   {i:2}. {gene}: {count:,}")

print(f"\nTOP 10 ANTIBIOTIC CLASSES")
top_classes = amr_full['resistance_class'].value_counts().head(10)
for i, (cls, count) in enumerate(top_classes.items(), 1):
    print(f"   {i:2}. {cls}: {count:,}")

print("\n" + "=" * 70)